# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [ ]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    usage = "Usage: greet <name> or add <x> <y>"
    if not argv:
        return usage
    if argv[0] == "greet" and len(argv) == 2:
        name = argv[1].strip()
        if not name:
            return usage
        return f"Hello, {name}!"
    if argv[0] == "add" and len(argv) == 3:
        try:
            return str(float(argv[1]) + float(argv[2]))
        except (ValueError, TypeError):
            return "Error: add requires two numeric values."
    return usage


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0.0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [ ]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    parser = argparse.ArgumentParser(description="Filter expense records")
    parser.add_argument("--category", help="Filter by category")
    parser.add_argument("--min", type=float, default=0, help="Minimum amount")
    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    if min_amount < 0:
        raise ValueError("Minimum amount must be non-negative.")
    filtered = []
    for expense in expenses:
        if category is not None and expense["category"].lower() != category.strip().lower():
            continue
        if expense["amount"] >= min_amount:
            filtered.append(expense)
    return filtered

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    if not expenses:
        return "No matching expenses."
    lines = [f"{'Date':<12} {'Category':<15} {'Amount':>10}  Note"]
    for expense in expenses:
        lines.append(
            f"{expense.get('date', ''):<12} {expense['category']:<15} "
            f"{expense['amount']:>10.2f}  {expense.get('note', '')}"
        )
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_data_20261005_224700.json` for `budget_data.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [ ]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w", encoding="utf-8") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    path = Path(filename)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{path.stem}_{timestamp}{path.suffix}"

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    source = Path(source_path)
    if not source.is_file():
        return None
    folder = Path(backup_dir)
    try:
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / make_backup_name(source.name)
        # Keep both copies if backups are requested within the same second.
        counter = 1
        original = destination
        while destination.exists():
            destination = folder / f"{original.stem}_{counter}{original.suffix}"
            counter += 1
        shutil.copy2(source, destination)
    except OSError as error:
        print(f"Backup failed: {error}")
        return None
    return str(destination)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [ ]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


In [ ]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    expenses = []
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row in reader:
                if None in row:
                    continue
                category = (row.get("category") or "").strip()
                if not category:
                    continue
                try:
                    amount = float(row.get("amount"))
                except (ValueError, TypeError):
                    continue
                expenses.append({
                    "date": row.get("date") or "",
                    "category": category,
                    "amount": amount,
                    "note": row.get("note") or "",
                })
    except (OSError, csv.Error) as error:
        print(f"Could not load expenses: {error}")
    return expenses

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    summary = {}
    for expense in expenses:
        category = expense["category"]
        summary[category] = summary.get(category, 0) + expense["amount"]
    return summary

def print_summary(summary):
    """Print a formatted category summary."""
    if not summary:
        print("No expenses to summarize.")
        return
    print(f"{'Category':<15} {'Total':>10}")
    for category, total in summary.items():
        print(f"{category:<15} {total:>10.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`

   (The exercise uses plain `python` to keep the strings short; on a real machine use the full path to your project's interpreter, as shown in the slides.)
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [ ]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    if not isinstance(minute, int) or isinstance(minute, bool) or not 0 <= minute <= 59:
        raise ValueError("Minute must be an integer from 0 to 59.")
    if not isinstance(hour, int) or isinstance(hour, bool) or not 0 <= hour <= 23:
        raise ValueError("Hour must be an integer from 0 to 23.")
    path = str(script_path)
    if " " in path:
        path = '"' + path + '"'
    return f"{minute} {hour} * * * python {path}"

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    if not isinstance(time_str, str) or len(time_str) != 5 or time_str[2] != ":":
        raise ValueError("Time must use HH:MM format.")
    hour, minute = time_str.split(":")
    if not hour.isdigit() or not minute.isdigit():
        raise ValueError("Time must use HH:MM format.")
    if not 0 <= int(hour) <= 23 or not 0 <= int(minute) <= 59:
        raise ValueError("Time must be between 00:00 and 23:59.")
    path = str(script_path)
    if " " in path:
        path = r'\"' + path + r'\"'
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {path}" /ST {time_str}'


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument. It comes **before** the subcommand: `["expenses_suite.json", "list"]`.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [ ]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w", encoding="utf-8") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


In [ ]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except (OSError, json.JSONDecodeError) as error:
        print(f"Could not load expenses: {error}")
        return []
    if not isinstance(data, list):
        print("Error: JSON must contain a list of expense dictionaries.")
        return []
    expenses = []
    for record in data:
        if not isinstance(record, dict):
            continue
        if not {"date", "category", "amount"}.issubset(record.keys()):
            continue
        category = record["category"]
        if not isinstance(category, str) or not category.strip():
            continue
        if isinstance(record["amount"], bool):
            continue
        try:
            amount = float(record["amount"])
        except (ValueError, TypeError):
            continue
        expenses.append({
            "date": str(record["date"]),
            "category": category.strip(),
            "amount": amount,
            "note": str(record.get("note", "")),
        })
    return expenses

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    parser = argparse.ArgumentParser(description="Expense automation suite")
    parser.add_argument("filename", help="JSON expense file")
    commands = parser.add_subparsers(dest="command", required=True)
    commands.add_parser("list", help="List expenses")
    commands.add_parser("summary", help="Show category totals")
    commands.add_parser("backup", help="Back up the JSON file")
    return parser

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    parser = build_suite_parser()
    args = parser.parse_args(argv)
    if args.command == "backup":
        destination = backup_file(args.filename)
        if destination is None:
            return "Error: could not back up the source file."
        return f"Backup created: {destination}"
    expenses = load_json_expenses(args.filename)
    if args.command == "list":
        return format_expenses(expenses)
    summary = summarize_by_category(expenses)
    if not summary:
        return "No expenses to summarize."
    lines = [f"{'Category':<15} {'Total':>10}"]
    for category, total in summary.items():
        lines.append(f"{category:<15} {total:>10.2f}")
    return "\n".join(lines)


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))
